# 2. La prédiction

Six modèles prédisent le salaire fixe ou le Compa-Ratio (salaire / milieu de grille), avec une régression linéaire, un arbre de décision ou un gradient boosting. À la fin, une validation croisée.

Ce notebook travaille sur le même tableau que `1_Visualisations` : les mêmes salariés, les mêmes colonnes. Les réglages sont dans `reglages.py` (anomalies à retirer, valeurs aberrantes, colonnes, réglages des modèles).

Pour changer un réglage : ouvrir `reglages.py`, le modifier, l'enregistrer, puis relancer ce notebook depuis sa première cellule.

---
## 1. Le tableau et les colonnes

Le même tableau que dans `1_Visualisations` : la détection tourne, les anomalies de `A_RETIRER` sortent, puis les valeurs aberrantes si `ENLEVER = True`. Les lignes affichées disent combien de salariés partent à chaque fois.

Les colonnes que les modèles reçoivent sont celles de la liste `COLONNES`.

In [ ]:
import importlib

import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, train_test_split

# Les trois fichiers communs du dossier : les réglages, la préparation du tableau, les modèles.
# importlib.reload les relit : un réglage changé dans reglages.py est pris en compte dès qu'on relance cette cellule.
import reglages, preparation, modeles
for fichier in (reglages, preparation, modeles):
    importlib.reload(fichier)
from preparation import CIBLES, RESULTATS, choisir_colonnes, outliers, preparer_tableau
from modeles import creer_modeles, metriques, salaire_predit

px.defaults.template = 'plotly_white'   # des graphes sur fond blanc
pd.set_option('display.precision', 2)   # les tableaux affichent 2 chiffres après la virgule ; les calculs gardent toute leur précision

tableau = preparer_tableau()
df = tableau.df   # les salariés gardés

# Les deux cibles et les colonnes de la liste COLONNES, en chiffres
donnees = choisir_colonnes(df)
# Les colonnes que les modèles reçoivent : celles de la liste, sans les deux cibles
X = donnees.drop(columns=CIBLES)

---
## 2. Les six modèles

| Modèle | Explique | Avec |
|---|---|---|
| M1 | le salaire | une régression linéaire |
| M2 | le salaire | un arbre de décision |
| M3 | le Compa-Ratio | une régression linéaire |
| M4 | le Compa-Ratio | un arbre de décision |
| M5 | le salaire | un gradient boosting |
| M6 | le Compa-Ratio | un gradient boosting |

Un arbre de décision seul ne sait donner qu'un petit nombre de réponses différentes : avec une profondeur de 5, 32 au plus, et tous les salariés qui tombent dans la même case reçoivent la même valeur. Le gradient boosting additionne des centaines de petits arbres, chacun corrigeant les erreurs des précédents : il donne une valeur propre à chaque salarié, et il suit des effets qui ne sont pas en ligne droite (une ancienneté qui compte plus dans certains niveaux, par exemple).

80 % des salariés servent à apprendre (train), 20 % à vérifier (test) : les mêmes pour les six modèles.

Les réglages de cette étape sont dans `reglages.py`.

Avec `LOG_SALAIRE = True`, le salaire est appris en logarithme : un écart se lit alors en pourcentage (10 % de plus), et les très hauts salaires pèsent moins. Avec `False`, il est appris tel quel.

Avec `LOG_COMPA_RATIO = True`, le Compa-Ratio est lui aussi appris en logarithme : quelques Compa-Ratio très hauts pèsent alors moins sur le modèle. Avec `False`, il est appris tel quel.

Les six modèles sont jugés sur le salaire, avec un seul résultat chacun :

* M1, M2 et M5 prédisent le salaire ; s'il est appris en logarithme, salaire prédit = exponentielle du log prédit ;
* M3, M4 et M6 prédisent le Compa-Ratio ; s'il est appris en logarithme, Compa-Ratio prédit = exponentielle du log prédit. Puis salaire prédit = Compa-Ratio prédit × `Mid`.

In [ ]:
# 1. On coupe en train et test, une seule fois. random_state : le même découpage à chaque lancement.
train, test = train_test_split(X.index, test_size=0.2, random_state=42)
print(f"Train : {len(train)} salariés. Test : {len(test)} salariés.")

# 2. Les six modèles (fichier modeles.py), avec les réglages de reglages.py
MODELES = creer_modeles()

# 3. Chaque modèle apprend sur le train, puis prédit pour les salariés du test, qu'il n'a jamais vus.
#    Un seul résultat par modèle, toujours en salaire (MAD). L'erreur est le vrai moins le prédit.
y_test, mid = df.loc[test, 'Fixe_Annuel_MAD'], df.loc[test, 'Mid']
resultats = pd.DataFrame({'y_test_MAD': y_test})
lignes = {}
for nom, (quoi, cible, modele) in MODELES.items():
    resultats[f'{nom}_y_pred_MAD'] = salaire_predit(modele, cible, train, test, X, df)
    resultats[f'{nom}_erreur_MAD'] = resultats['y_test_MAD'] - resultats[f'{nom}_y_pred_MAD']
    # Les métriques du modèle, sur le test : MAE et RMSE sur le salaire, R² sur ce que le modèle prédit
    lignes[f'{nom} : {quoi}'] = metriques(cible, y_test, resultats[f'{nom}_y_pred_MAD'], mid)
    # Un salaire prédit négatif ou nul est impossible : c'est le signe qu'une valeur aberrante est restée dans les données
    impossibles = int((resultats[f'{nom}_y_pred_MAD'] <= 0).sum())
    if impossibles:
        print(f"ATTENTION : {nom} prédit {impossibles} salaire(s) négatif(s) ou nul(s). Des valeurs aberrantes sont restées dans les données : essayer ENLEVER = True dans reglages.py.")
display(resultats.head(10).round(2))

# 4. Les métriques des six modèles, sur le même test
tableau_metriques = pd.DataFrame(lignes).T
display(tableau_metriques.round(2))

* **MAE** : l'erreur moyenne, dans l'unité du salaire.
* **RMSE** : pareil, mais les grosses erreurs comptent plus.
* **R²** : un seul par modèle, calculé sur ce que le modèle prédit. Pour M1, M2 et M5, c'est le salaire : il est haut dès que le salaire suit la grille, même si le modèle n'apprend rien d'autre. Pour M3, M4 et M6, c'est le Compa-Ratio : la part de l'écart autour du `Mid` que le modèle explique.

Le R² de M1, M2 et M5 ne se compare donc pas à celui de M3, M4 et M6. Pour comparer les six modèles, prendre la MAE et la RMSE : elles sont en MAD pour tous.

M5 et M6 répondent à une question : le salaire dépend-il des colonnes d'une façon qu'une ligne droite ne suit pas ? Si leur MAE est nettement plus basse que celle de M1 et de M3, oui. Si elle est presque la même, la limite vient des colonnes, pas du modèle.

In [ ]:
# Le salaire prédit face au vrai, pour les six modèles : un bon modèle met les points le long de la diagonale
points = pd.concat([pd.DataFrame({'Modèle': f'{nom} : {quoi}', 'y_test': resultats['y_test_MAD'],
                                  'y_pred': resultats[f'{nom}_y_pred_MAD'], 'erreur': resultats[f'{nom}_erreur_MAD']})
                    for nom, (quoi, cible, modele) in MODELES.items()])
fig = px.scatter(points, x='y_test', y='y_pred', facet_col='Modèle', facet_col_wrap=2, opacity=0.4, height=1300,
                 title='Salaire fixe prédit face au vrai (test)',
                 labels={'y_test': 'Vrai salaire fixe (y_test)', 'y_pred': 'Salaire fixe prédit (y_pred)'})
bornes = [points['y_test'].min(), points['y_test'].max()]
fig.add_scatter(x=bornes, y=bornes, mode='lines', line=dict(color='black'), showlegend=False, row='all', col='all')
fig.for_each_annotation(lambda a: a.update(text=a.text.replace('Modèle=', '')))
fig.show()

# Les erreurs sur le salaire : un bon modèle les serre autour de 0
fig = px.histogram(points, x='erreur', facet_col='Modèle', facet_col_wrap=2, nbins=60, height=1000,
                   title='Répartition des erreurs sur le salaire fixe (test)', labels={'erreur': 'Erreur (y_test − y_pred)'})
fig.add_vline(x=0, line_color='black')
fig.for_each_annotation(lambda a: a.update(text=a.text.replace('Modèle=', '')))
fig.show()

In [ ]:
# Export général en Excel : la base complète des salariés du test (toutes les colonnes du tableau, matricule compris),
# avec à côté le vrai salaire (y_test), puis le salaire prédit (y_pred) et l'erreur de chacun des six modèles.
export = df.loc[resultats.index].join(resultats).round(2)   # 2 chiffres après la virgule dans le fichier
# Le fichier s'écrit dans le dossier resultats
export.to_excel(RESULTATS / 'resultats_prediction.xlsx', index=False)
print(f"{len(export)} salariés du test et {export.shape[1]} colonnes exportés dans resultats/resultats_prediction.xlsx")
export.head()

---
## 3. La validation croisée

À l'étape 2, les modèles sont jugés sur un seul test : 20 % des salariés, tirés une fois. Avec un autre tirage, les chiffres bougeraient un peu.

Ici, les salariés sont coupés en 5 paquets (réglage `PAQUETS`). Chaque paquet sert de test à son tour, pendant que les modèles apprennent sur les 4 autres. Chaque salarié est donc prédit une fois, par un modèle qui ne l'a pas vu.

Le reste ne change pas : les mêmes salariés, les mêmes six modèles, les mêmes réglages.

Lecture du tableau : chaque case est la moyenne des 5 paquets. C'est le chiffre à retenir, plus sûr que celui d'un seul test.

La validation croisée ne rend pas les modèles meilleurs. Elle donne des chiffres plus sûrs.

In [ ]:
# 1. On coupe tous les salariés en paquets de même taille. random_state : les mêmes paquets à chaque lancement.
decoupage = KFold(n_splits=reglages.PAQUETS, shuffle=True, random_state=42)

# 2. Chaque paquet sert de test à son tour, les autres paquets servent à apprendre
lignes = []
for numero, (apprend, verifie) in enumerate(decoupage.split(X), start=1):
    # KFold rend des positions (0, 1, 2...) : on retrouve les lignes du tableau, comme train et test à l'étape 2
    apprend, verifie = X.index[apprend], X.index[verifie]
    y_vrai, mid_paquet = df.loc[verifie, 'Fixe_Annuel_MAD'], df.loc[verifie, 'Mid']
    for nom, (quoi, cible, modele) in MODELES.items():
        y_pred = salaire_predit(modele, cible, apprend, verifie, X, df)
        # Les mêmes métriques qu'à l'étape 2, calculées sur ce paquet
        lignes.append({'Modèle': f'{nom} : {quoi}', 'Paquet': numero, **metriques(cible, y_vrai, y_pred, mid_paquet)})
# Une ligne par modèle et par paquet
par_paquet = pd.DataFrame(lignes)

# 3. Le même tableau qu'à l'étape 2, avec la moyenne des paquets dans chaque case
validation = par_paquet.groupby('Modèle', sort=False)[['MAE (MAD)', 'RMSE (MAD)', 'R²']].mean()
display(validation.round(2))